# Performances of 2D integration vs 1D integration

This is dependent on:
* Number of azimuthal bins
* Pixel splitting
* Algorithm
* Implementation (i.e. programming language)
* Hardware used

Thus there is no general answer. But here is a quick benchmark to evaluate the penalty on performances:

import sys
import os
import time
import numpy
import fabio
import pyFAI
from pyFAI.test.utilstest import UtilsTest
import pyFAI.method_registry
import pyFAI.integrator.azimuthal
print(f"Python version: {sys.version}")
print(f"PyFAI version: {pyFAI.version}")
start_time = time.perf_counter()

In [1]:
import sys
import os
import time

os.environ["PYOPENCL_COMPILER_OUTPUT"] = "0"
start_time = time.perf_counter()

In [2]:
import fabio
import pyFAI
from pyFAI.test.utilstest import UtilsTest
import pyFAI.method_registry
import pyFAI.integrator.azimuthal
print(f"Python version: {sys.version}")
print(f"PyFAI version: {pyFAI.version}")


Python version: 3.14.0 | packaged by conda-forge | (main, Oct 22 2025, 23:24:08) [GCC 14.3.0]
PyFAI version: 2026.7.0-dev0


In [3]:
print("Number of way to performing integration:", len(pyFAI.method_registry.IntegrationMethod.list_available()))

Number of way to performing integration: 95


In [4]:
ai = pyFAI.load(UtilsTest.getimage("Pilatus1M.poni"))
img = fabio.open(UtilsTest.getimage("Pilatus1M.edf")).data
ai

Detector Pilatus 1M	 PixelSize= 172µm, 172µm	 BottomRight (3)
Wavelength= 1.000000 Å
SampleDetDist= 1.583231e+00 m	PONI= 3.341702e-02, 4.122778e-02 m	rot1=0.006487  rot2=0.007558  rot3=0.000000 rad
DirectBeamDist= 1583.310 mm	Center: x=179.981, y=263.859 pix	Tilt= 0.571° tiltPlanRotation= 130.640° λ= 1.000Å

In [5]:
%%time
#Tune those parameters to match your needs:
kw1 = {"data": img, "npt":1000}
kw2 = {"data": img, "npt_rad":1000}
#Actual benchmark:
res = {}
for k,v in pyFAI.method_registry.IntegrationMethod._registry.items():
    print(k)
    if k.dim == 1:
        res[k] = %timeit -o ai.integrate1d(method=v, **kw1)
    else:
        res[k] = %timeit -o ai.integrate2d(method=v, **kw2)

Method(dim=1, split='no', algo='histogram', impl='python', target=None)


30.9 ms ± 400 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
Method(dim=2, split='no', algo='histogram', impl='python', target=None)


141 ms ± 195 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
Method(dim=1, split='no', algo='histogram', impl='cython', target=None)


11.2 ms ± 25.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Method(dim=2, split='no', algo='histogram', impl='cython', target=None)


16.6 ms ± 11.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Method(dim=1, split='bbox', algo='histogram', impl='cython', target=None)


...

449 ms ± 36.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
Method(dim=1, split='full', algo='lut', impl='opencl', target=(1, 0))


5.1 ms ± 123 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Method(dim=2, split='full', algo='lut', impl='opencl', target=(1, 0))


325 ms ± 62.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
Method(dim=1, split='full', algo='lut', impl='opencl', target=(2, 0))


5.03 ms ± 375 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
Method(dim=2, split='full', algo='lut', impl='opencl', target=(2, 0))


434 ms ± 61.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
CPU times: user 1h 30min 10s, sys: 2min 32s, total: 1h 32min 43s
Wall time: 7min 50s


In [6]:
print("-"*80)
print(f"{'Split':5s} | {'Algo':9s} | {'Impl':6s}| {'1d (ms)':8s} | {'2d (ms)':8s} | {'ratio':6s} | Device")
print("-"*80)
for k in res:
    if k.dim == 1:
        k1 = k
        k2 = k._replace(dim=2)
        if k2 in res:
            print(f"{k1.split:5s} | {k1.algo:9s} | {k1.impl:6s}| {res[k1].best*1000:8.3f} | {res[k2].best*1000:8.3f} | {res[k2].best/res[k1].best:6.1f} | ",
                    end="")
        if k.target:
            print(pyFAI.method_registry.IntegrationMethod._registry.get(k).target_name)
        else:
            print()
print("-"*80)

--------------------------------------------------------------------------------
Split | Algo      | Impl  | 1d (ms)  | 2d (ms)  | ratio  | Device
--------------------------------------------------------------------------------
no    | histogram | python|   30.355 |  140.383 |    4.6 | 
no    | histogram | cython|   11.173 |   16.570 |    1.5 | 
bbox  | histogram | cython|   26.029 |   32.514 |    1.2 | 
full  | histogram | cython|  154.104 |  285.141 |    1.9 | 
no    | csr       | cython|   10.149 |   12.338 |    1.2 | 
bbox  | csr       | cython|    8.158 |   13.813 |    1.7 | 
no    | csr       | python|   10.459 |   14.941 |    1.4 | 
bbox  | csr       | python|   13.828 |   17.852 |    1.3 | 
no    | csc       | cython|    8.174 |   10.521 |    1.3 | 
bbox  | csc       | cython|   10.411 |   14.030 |    1.3 | 
no    | csc       | python|   11.349 |   14.777 |    1.3 | 
bbox  | csc       | python|   15.129 |   22.268 |    1.5 | 
bbox  | lut       | cython|    9.734 |   11.890 |   

In [7]:
print(f"Total runtime: {time.perf_counter()-start_time:.3f}s")

Total runtime: 472.226s
